# 02 · Agent Patterns

This notebook covers four agent-orchestration patterns 

* Plan-and-Solve
* Route-and-Solve
* ToolRAG
* ReAct

You'll learn how each one solves a different limitation of the basic function-calling loop from [`01_function_calling.ipynb`](01_function_calling.ipynb): 

breaking a request into a plan you can inspect and revise, splitting a large toolset across specialized subagents, retrieving only the tools relevant to a query instead of binding hundreds of them at once, and making the model's reasoning visible step by step.

By the end, you'll be able to recognize which pattern fits a given problem and explain the tradeoff each one makes for that benefit -- more moving parts, an extra routing call, a new retrieval failure mode, or more verbose output. Every pattern section reuses the same `llm`, `get_stock_price` and `get_current_weather` from `granite_agent` (built in [`01_function_calling.ipynb`](01_function_calling.ipynb)) instead of redefining them.

### Which pattern for which problem?

| Pattern | Core idea | Good fit when |
| --- | --- | --- |
| **Plan-and-Solve** | Plan all the steps up front, execute them, replan from what actually happened | The task is multi-step with real dependencies between steps, and you want the plan itself to be inspectable |
| **Route-and-Solve** | A router picks one specialized subagent (with its own, smaller toolset) per query | Your tools naturally group into domains and a single agent seeing all of them would be error-prone |
| **ToolRAG** | Semantically retrieve a small relevant subset of tools before binding them to the model | You have hundreds or thousands of tools -- too many to fit in the model's context at once |
| **ReAct** | Interleave visible reasoning ("Thought") with tool calls, in one loop | You want the model's reasoning trace to be inspectable step by step, not just its tool calls |


# Steps

## Step 1. Set up your environment
You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

## Step 2. Set up a Granite AI model instance

This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://ollama.com) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

* See [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb) for information on getting ready to use Replicate. 
* See [Getting Started with Ollama](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Ollama.ipynb) for information on getting ready to use Ollama.

Prior to running this notebook, you should have either started a local Ollama server on your computer, or setup Replicate access and obtained an [API token](https://replicate.com/account/api-tokens).

## Step 3. Install relevant libraries and set up credentials and the Granite model

We'll need a few libraries across the four patterns in this notebook: LangGraph and LangChain for all of them, plus Chroma and HuggingFace embeddings for ToolRAG, and matplotlib/pandas for Plan-and-Solve's forecast plot.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    langchain_huggingface sentence_transformers \
    chromadb langchain-chroma \
    matplotlib pandas \
    grandalf
! echo "::endgroup::"

Now we will create the LangChain object to use the Granite model, and import the tools -- the exact same `get_llm()`, `get_stock_price` and `get_current_weather` built in [`01_function_calling.ipynb`](01_function_calling.ipynb). Every pattern below reuses these instead of redefining them.

In [ ]:
import sys
from pathlib import Path

# Make the reusable `granite_agent` package (built in 01_function_calling) importable.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "granite_agent").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from granite_agent.model import get_llm
from granite_agent.tools import get_current_weather, get_stock_price

llm = get_llm()
print(f"Connected via {type(llm).__name__}, model={getattr(llm, 'model', None)!r}")

## Step 4: Define the functions

`get_stock_price` and `get_current_weather` were already imported above -- no need to redefine them here, which keeps a single source of truth for both tools across every pattern below.

Each pattern section below still defines its own **pattern-specific** tools (a router's finance/weather subagents, Plan-and-Solve's forecast/plotting tools, ToolRAG's demo tool pool, ReAct's travel tools) locally, since those aren't reused anywhere else.

## Step 5: Pattern -- Plan-and-Solve ⭐

This section demonstrates a **Plan-Solve Agent** that can break down complex tasks into executable steps, execute them using a function-calling agent, and adapt the plan based on results.

A [Plan-Solve Agent](https://arxiv.org/abs/2305.04091) pattern follows these steps:

1. **Planning Phase**: The agent analyzes a user request and creates a step-by-step plan
2. **Execution Phase**: The agent executes each step using function calling with available tools
3. **Replanning Phase**: The agent reviews progress and updates the plan based on results
4. **Iteration**: The cycle continues until the task is complete

This approach allows the agent to handle multi-step tasks by:

- Breaking down complex requests into manageable steps
- Using external tools to gather information
- Adapting the plan based on intermediate results
- Coordinating multiple tool calls to achieve the final goal

This pattern allows a more structured task breakdown with stronger error recovery by separating the planning phase from acting, and avoids premature, incomplete answers.

This section leverages IBM [granite-4.2-8b](https://huggingface.co/ibm-granite/granite-4.2-8b) rather than the `3b` default used elsewhere in this notebook, since it features improved instruction following (IF) and tool-calling capabilities that suit the Plan-Solve pattern well.

**Execution Flow:**
1. `planner_node` analyzes the user request and creates an initial structured plan
2. `function_calling_agent` executes the plan steps, invoking tools as needed, based on the pattern from [`01_function_calling.ipynb`](01_function_calling.ipynb)
3. Results return to `replan` which reviews progress and updates the plan
4. The cycle continues (replan → function_calling_agent) until all steps are complete or `plan_complete` is called

### Use the larger model for this section

The planner and replanner prompts are long (they embed the full tool list and, on replans, the execution history), and this pattern benefits from stronger instruction-following than `3b` reliably gives. This uses `get_llm()` again -- reused exactly as in Step 2 -- just with a different model name.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

llm_plan = get_llm("ibm-granite/granite-4.2-30b")
print(f"Connected via {type(llm_plan).__name__}, model={getattr(llm_plan, 'model', None)!r}")

### Required credentials

If using Replicate, you need to set the environment variable `REPLICATE_API_TOKEN` to your Replicate API key.

`get_current_weather` (imported above) uses a `WEATHER_API_KEY`. To generate one, please [create an account](https://home.openweathermap.org/users/sign_up). Upon creating an account, select the "API Keys" tab to display your free key. The two new tools below (`get_geo_coordinates`, `get_weather_forecast`) use the same key -- nothing new to configure if you already set it up for 01.

**If you run this notebook in Colab, store these private keys as Colab Secrets.**
**If you run this notebook locally, store these private keys in a `.env` file in the same directory as this notebook.**

In [ ]:
import operator
from datetime import datetime
from typing import Annotated, TypedDict

from IPython.display import Image, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from langchain_core.messages import AnyMessage, AIMessage, HumanMessage, SystemMessage, ToolCall, ToolMessage, filter_messages
from langchain_core.tools import tool
from langchain_core.utils.utils import convert_to_secret_str
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

WEATHER_API_KEY = convert_to_secret_str(get_env_var("WEATHER_API_KEY", "unset"))

### Define planner prompts

The Plan-Solve pattern requires two types of prompts:

- **Planner Prompt**: Creates the initial step-by-step plan by analyzing the user's request and available tools
- **Replanner Prompt**: Reviews execution results and updates the plan, removing completed steps and adding new ones as needed

Both prompts instruct the LLM to return structured JSON plans with ordered steps.

In [ ]:
planner_prompt = """You are a task planner agent. For context, today's date is {date}.
You will be provided a user request with an objective, your goal is to create a to do list which consists in a step by step plan.
This plan should involve individual tasks, that if executed correctly will yield the correct answer. Do not add any superfluous steps.
The result of the final step should be the plan_complete. Make sure that each step has all the information needed and tool dependencies are managed properly - do not skip steps

<Available Tools>
You have access to these tools:
1. **get_current_weather**: Fetches the current weather for a given location
2. **get_geo_coordinates**:  Retrieves geographic coordinates (latitude and longitude) for a specified city.
3. **get_weather_forecast**: Retrieves a 5-day weather forecast for a specific latitude and longitude.
4. **plot_weather_timeseries**: creates a time series plot from weather forecast data obtained from get_weather_forecast, supporting multiple series.
5. **plan_complete**: Call this tool to complete the plan
</Available Tools>


provide a plan as an ordered list of steps, where each step is a JSON object following the schema:

{{"steps": [ {{"description": (natural language high level description of the task - you can only use an available tool)}},...]}}

Return only the JSON plan with steps key and list of steps:

"""


replanner_prompt = """ You are a task planner agent that can delegate task to tools at your disposal. \
You will be provided :

1- a user request with an objective.
2- an initial plan that you created which consists in a list of tool calls.
3- the result of the steps executed so far from the initial plan and their results.

your goal is to review the results of steps executed from the initial plan and propose an updated version of the plan. \
Steps already successfully completed should not be included in the new plan.
New steps plan proposed must repeat ALL data required in their description.
if all steps were completed return a new single steps plan with only plan_complete tool

the user request and objective was this:

{input}

Your original plan to fulfill the user request was this:

{plan}

You have currently done the follow steps:

{past_steps}

Update your plan accordingly. If no more steps are needed and you can return to the user plan_complete. Otherwise.
Only add steps to the plan that still NEED to be done.
if all steps were completed return a new single steps plan with only plan_complete tool,do not return previously done steps as part of the updated plan.

<Available Tools>
You have access to these tools:
1. **get_current_weather**: Fetches the current weather for a given location
2. **get_geo_coordinates**:  Retrieves geographic coordinates (latitude and longitude) for a specified city.
3. **get_weather_forecast**: Retrieves a 5-day weather forecast for a specific latitude and longitude.
4. **plot_weather_timeseries**: creates a time series plot from weather forecast data obtained from get_weather_forecast, supporting multiple series.
5. **plan_complete**: Call this tool to complete the plan

</Available Tools>


provide an updated plan as an ordered list of the remaining steps needed to complete the user request and objective, where each step is a JSON object following the schema:

{{"steps": [ {{"description": (natural language description of the task with dependencies from previous steps - data required must be included in step description - you can only use an available tool)}},...]}}

Return only the JSON plan with steps key and list of steps:


"""

### Define agent state

The Plan-Solve Agent uses typed state to manage data flow through the graph:

- **Plan**: Represents a structured plan with ordered steps returned by the planning LLM
- **PlanSolve**: Coordinates the planning cycle, tracking the current plan, execution history, and whether planning is complete

These state definitions enable LangGraph to automatically merge state updates as the agent progresses through planning and execution phases.

In [ ]:
class Plan(BaseModel):
    """Plan to follow created by planner"""
    steps: list[str] = Field(description="plan steps to be executed in sorted order")


class PlanSolve(TypedDict, total=False):
    """
    State for the planner.
    Manages coordination between planner and executor for tracking progress and replan.
    """
    input: str
    plan: list[str]
    past_steps: Annotated[list[tuple[list[ToolCall], str]], operator.add]
    plan_completed: bool

### Define the pattern-specific tools

The Plan-Solve Agent has access to a focused toolkit for weather analysis and data visualization:

- **get_geo_coordinates**: Converts city names to latitude/longitude coordinates
- **get_current_weather**: Fetches current weather conditions for a location (imported at the top of this notebook)
- **get_weather_forecast**: Retrieves 5-day forecast data for specific coordinates
- **plot_weather_timeseries**: Creates time series visualizations comparing multiple weather forecasts
- **plan_complete**: Signals completion of all plan steps

Some tools use internal helper functions (prefixed with `_internal`) to separate LangChain tool interfaces from implementation logic. Each tool includes detailed docstrings that help the planning LLM understand when and how to invoke them.

In [ ]:
@tool(parse_docstring=True)
def plan_complete() -> str:
    """
    Complete the plan executing with END node
    """
    print("***[PLAN_COMPLETE] TOOL CALLED***")
    return "Plan execution completed successfully"

def _get_geo_coordinates_internal(city_name: str, state_code: str, country: str) -> tuple[float, float]:
    """
    Internal function to get geographic coordinates (not a LangChain tool).
    """
    print(f"Getting geo coordinates data for {city_name} {state_code} {country}")
    apikey = WEATHER_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return 37.7790262, -122.419906

    try:
        geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city_name},{state_code},{country}&limit=5&appid={apikey}"
        geo_data = requests.get(geo_url)
        data = geo_data.json()

        return data[0].get("lat", 37.7790262), data[0].get("lon", -122.419906)
    except Exception as e:
        print(f"Error fetching weather data: {e}")
        return 37.7790262, -122.419906


def _get_weather_forecast_internal(lat: float, lon: float) -> list:
    """
    Internal function to get weather forecast (not a LangChain tool).
    """
    print(f"Getting weather forecast for {lat} {lon} ")
    apikey = WEATHER_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return [{"2025-10-04 12:00:00": 25.3}]

    try:
        weather_url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={apikey}&units=metric"
        weather_data = requests.get(weather_url)
        data = weather_data.json()

        weather_list = data.get("list", [])
        formatted_data = []

        for item in weather_list:
            dt = item.get("dt", "")
            temperature = item.get("main", {}).get("temp", 0)

            if dt and temperature is not None:
                dt_datetime = datetime.fromtimestamp(dt)
                dt_string = dt_datetime.strftime("%Y-%m-%d %H:%M:%S")
                formatted_data.append({dt_string: temperature})
        print(f"{len(formatted_data)} forecast datapoints fetched:")
        return formatted_data
    except Exception as e:
        raise e


def _plot_weather_timeseries_internal(
    weather_data: dict[str, list[dict[str, float]]], title: str = "Weather Forecast", save_path: str | None = None
) -> None:
    """
    Internal function to plot weather timeseries (not a LangChain tool).
    """
    if not weather_data:
        raise ValueError("Weather data cannot be empty")

    dataframes = []
    all_series_labels = []

    for series_name, series_list in weather_data.items():
        if not series_list:
            continue

        datetimes = []
        temperatures = []

        for item in series_list:
            if not isinstance(item, dict) or len(item) != 1:
                raise ValueError("Each item in weather_data must be a dictionary with exactly one key-value pair")

            dt_str = list(item.keys())[0]
            temp = list(item.values())[0]

            try:
                # pd.to_datetime tolerates the format variations models tend to
                # produce (missing seconds, "T"/"Z" separators, etc.), unlike a
                # single fixed strptime format.
                dt_obj = pd.to_datetime(dt_str)
                datetimes.append(dt_obj)
                temperatures.append(temp)
            except (ValueError, TypeError) as e:
                raise ValueError(f"Invalid datetime format in data: {dt_str}. Expected \"YYYY-MM-DD HH:MM:SS\"") from e

        if datetimes:
            df = pd.DataFrame({"datetime": datetimes, series_name: temperatures})
            df.set_index("datetime", inplace=True)
            dataframes.append(df)
            all_series_labels.append(series_name)

    if not dataframes:
        raise ValueError("No valid datetime-temperature pairs found in weather_data")

    if len(dataframes) == 1:
        merged_df = dataframes[0]
    else:
        merged_df = dataframes[0]
        for df in dataframes[1:]:
            merged_df = merged_df.join(df, how="outer")

    plt.figure(figsize=(14, 8))

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#7209B7", "#2D5016"]
    markers = ["o", "s", "^", "D", "v", "p"]

    for i, (series_name, series_data) in enumerate(merged_df.items()):
        if series_data.dropna().empty:
            continue

        color = colors[i % len(colors)]
        marker = markers[i % len(markers)]

        plt.plot(
            series_data.index,
            series_data.values,
            marker=marker,
            linewidth=2,
            markersize=6,
            color=color,
            label=series_name,
            alpha=0.8,
        )

    plt.title(title, fontsize=16, fontweight="bold", pad=20)
    plt.xlabel("Date and Time", fontsize=12)
    plt.ylabel("Temperature (°C)", fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=6))
    plt.xticks(rotation=45)

    if len(merged_df.columns) > 1:
        plt.legend(loc="best", framealpha=0.9)

    all_temps = merged_df.values.flatten()
    all_temps = all_temps[~np.isnan(all_temps)]
    if len(all_temps) > 0:
        min_temp = np.min(all_temps)
        max_temp = np.max(all_temps)
        plt.text(
            0.02,
            0.98,
            f"Range: {min_temp:.1f}°C - {max_temp:.1f}°C",
            transform=plt.gca().transAxes,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
        )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    else:
        plt.show()


@tool(parse_docstring=True)
def get_geo_coordinates(city_name: str, state_code: str, country: str) -> tuple[float, float]:
    """
    Retrieves geographic coordinates (latitude and longitude) for a specified city.

    This function converts city names into precise geographic coordinates that can be used
    for weather API calls and other location-based services. It uses the OpenWeatherMap
    Geocoding API to resolve city names to coordinates.

    Args:
        city_name (str): The name of the city. Examples "New York", "Montréal", "London"
        state_code (str): The state or province code. Examples "NY", "CA", "Québec", "ON"
        country (str): The two-letter country code. Examples "US", "CA", "GB", "FR"

    Returns:
        tuple[float, float]: A tuple containing (latitude, longitude) coordinates.
            - Latitude: Decimal degrees between -90 and 90 (negative = South)
            - Longitude: Decimal degrees between -180 and 180 (negative = West)

    Raises:
        Exception: If API request fails, returns fallback coordinates for San Francisco

    Note:
        - Returns the first matching result from the geocoding API
        - If no API key is configured, returns San Francisco coordinates as fallback
    """
    return _get_geo_coordinates_internal(city_name, state_code, country)


@tool(parse_docstring=True)
def get_weather_forecast(lat: float, lon: float) -> list:
    """
    Retrieves a 5-day weather forecast for a specific location and date.

    This function fetches detailed weather forecast data for the next 5 days starting from
    the specified datetime string. The forecast includes 3-hourly intervals with temperature data
    formatted for easy consumption by AI agents and applications.

    Args:
        lat: Latitude coordinate in decimal degrees. Range: -90 to 90 Examples 40.7128 (New York), 45.5017 (Montréal)
        lon: Longitude coordinate in decimal degrees. Range: -180 to 180 Examples -74.0060 (New York), -73.5673 (Montréal)

    Returns:
        list: A list of dictionaries, each containing a datetime-temperature pair.
              Format: [{"YYYY-MM-DD HH:MM:SS": temperature_in_celsius}]

    Raises:
        ValueError: If start_datetime is not in the correct format
        Exception: If API request fails, raises the original exception

    Note:
        - Temperature values are in Celsius
        - If no API key is configured, returns demo data for demonstration
        - Use get_geo_coordinates() to convert city names to lat/lon coordinates
    """
    return _get_weather_forecast_internal(lat, lon)


@tool(parse_docstring=True)
def plot_weather_timeseries(weather_data: dict[str, list[dict[str, float]]], title: str = "Weather Forecast") -> str:
    """
    Creates a time series plot from weather forecast data, supporting multiple series.
    expected datetime format is "YYYY-MM-DD HH:MM:SS" for weather_data

    This function can multiple forecasts on the same plot.
    It uses pandas to merge multiple time series by datetime, allowing comparison of different
    forecasts or locations.

    Args:
        weather_data: dict[str, list[dict[str, float]]]: Dictionary with city names as keys and lists of datetime-temperature pairs as values. Format: {"CityName": [{"YYYY-MM-DD HH:MM:SS": temperature}, ...]}
        title: Title for the plot. Defaults to "Weather Forecast"

    Returns:
        str: Completion message

    Raises:
        ValueError: If weather_data is empty or has invalid format
        Exception: If plotting fails
    """
    _plot_weather_timeseries_internal(weather_data=weather_data, title=title, save_path=None)
    return f"Plot {title} generated"

### Build the function-calling agent

This is the same `create_agent` pattern from [`01_function_calling.ipynb`](01_function_calling.ipynb), just with this section's own tool list. This graph will be invoked by the Plan-Solve agent to execute individual steps from the plan.

In [ ]:
from langchain.agents import create_agent

plan_tools = [get_current_weather, get_weather_forecast, plot_weather_timeseries, get_geo_coordinates, plan_complete]

fc_agent = create_agent(
    name="fc_agent",
    model=llm_plan,
    tools=plan_tools,
)

### Define the Plan-Solve agent nodes

The Plan-Solve agent orchestrates the planning-execution cycle through four key nodes:

- **planner_node**: Analyzes the user's request and generates an initial structured plan with ordered steps
- **execute_step**: Delegates plan steps to the function calling agent and captures execution results
- **replanner_node**: Reviews execution results, removes completed steps, and updates the plan based on progress
- **should_end**: Conditional edge that determines whether to continue the cycle or complete the workflow

This implements the core Plan-Solve pattern: Plan → Execute → Replan → Execute → ... until complete.

In [ ]:
def get_today_str() -> str:
    """Get current date in a human-readable format."""
    # Format date without leading zero for day (cross-platform compatible)
    date = datetime.now()
    day = date.day  # Remove leading zero
    return date.strftime(f"%a %b {day}, %Y")


def planner_node(state: PlanSolve) -> PlanSolve:
    """Generate an initial plan to solve the user's request."""
    print("***[PLANNER] NODE***")
    input = state.get("input")
    system_message = planner_prompt.format(date=get_today_str())

    structured_planner_llm = llm_plan.with_structured_output(Plan, method="json_schema")

    plan: Plan = structured_planner_llm.invoke([SystemMessage(content=system_message), HumanMessage(content=input)])  # type: ignore
    print("\n".join(f"{i}. {step}" for i, step in enumerate(plan.steps, start=1)))
    return PlanSolve(plan=plan.steps, plan_completed=False)


def replanner_node(state: PlanSolve) -> PlanSolve:
    """Update the plan based on previous execution results and progress."""
    print("***[REPLANNER] NODE***")
    input = state.get("input")
    plan = state.get("plan", [])
    past_steps = state.get("past_steps", [])
    plan_str = "\n".join(f"{i}. {step}" for i, step in enumerate(plan, start=1))
    past_steps_str = "\n".join(f"{i}. {step[0]} and result was \n{step[1]}\n\n" for i, step in enumerate(past_steps, start=1))

    system_message = replanner_prompt.format(input=input, plan=plan_str, past_steps=past_steps_str)

    structured_replanner_llm = llm_plan.with_structured_output(Plan, method="json_schema")

    replan: Plan = structured_replanner_llm.invoke([SystemMessage(content=system_message)])  # type: ignore

    return PlanSolve(plan=replan.steps)


def should_end(state: PlanSolve):
    """Conditional edge to determine whether the planning-execution cycle should terminate."""
    if state.get("plan_completed", False) or not state.get("plan"):
        return END
    else:
        return "function_calling_agent"


def execute_step(state: PlanSolve) -> PlanSolve:
    """Execute a step from the plan using the function-calling agent."""
    print("***[EXECUTE STEP] NODE***")
    plan = state.get("plan", [])
    plan_str = "\n".join(f"{i}. {step}" for i, step in enumerate(plan, start=1))
    task_formatted = f"""For the following plan: {plan_str}\n\nYou are tasked with executing these steps above."""
    messages: list[AnyMessage] = [HumanMessage(content=task_formatted)]
    # Invoke fc_agent outside LangGraph subgraph context using a thread
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(fc_agent.invoke, {"messages": messages})
        agent_response = future.result()
    plan_completed = False

    ai_messages: list[AIMessage] = filter_messages(agent_response.get("messages"), include_types=[AIMessage])  # type: ignore
    for ai_msg in ai_messages:
        if ai_msg.tool_calls:
            for tool_call in ai_msg.tool_calls:
                if tool_call["name"] == "plan_complete":
                    plan_completed = True
                    print("***[EXECUTE STEP] plan_complete tool detected***")
                    break
    tool_messages: list[ToolMessage] = filter_messages(agent_response.get("messages"), include_types=[ToolMessage])  # type: ignore
    past_steps = [
            (ai_m.tool_calls, str(tc_m.text))
            for ai_m, tc_m in zip(
                ai_messages,
                tool_messages,
            )
        ]
    return PlanSolve(past_steps=past_steps, plan_completed=plan_completed)

### Assemble the Plan-Solve agent graph

Now we assemble the complete Plan-Solve agent by connecting all nodes into a LangGraph workflow:

- **Flow**: START → planner_node → function_calling_agent → replan → (conditional routing)
- **Planning cycle**: After replanning, the graph either continues back to execute more steps or ends if the plan is complete
- **Adaptive execution**: Each cycle allows the agent to adapt its plan based on tool execution results

This creates a dynamic agent that can break down complex tasks, execute them step-by-step, and adjust its approach based on intermediate results.

In [ ]:
graph = StateGraph(PlanSolve)

graph.add_node("planner_node", planner_node)
graph.add_node("function_calling_agent", execute_step)
graph.add_node("replan", replanner_node)

graph.add_edge(START, "planner_node")
graph.add_edge("planner_node", "function_calling_agent")
graph.add_edge("function_calling_agent", "replan")

graph.add_conditional_edges(
    "replan",
    should_end,
    {
        "function_calling_agent": "function_calling_agent",
        END: END,
    },
)

planner_agent = graph.compile()

We can visualize the compiled graph to see how the Plan-Solve pattern orchestrates planning and execution.

In [ ]:
try:
    display(Image(planner_agent.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

### Run it

Now let's run the Plan-Solve Agent with a complex weather forecast comparison task. This will demonstrate how the agent:

1. **Plans**: Breaks down the request into steps
2. **Executes**: Uses tools to gather data
3. **Replans**: Adapts based on results
4. **Completes**: Generates the final visualization

In [ ]:
user_input = "compare weather forecast for Yorktown and Armonk"

print("Starting Plan-Solve Agent execution...")
print(f"User request: {user_input}")

result = planner_agent.invoke({"input": user_input})

print("Plan-Solve Agent execution completed!")

**Plan-and-Solve recap:** Planning & Replanning gives you an initial plan you can inspect before anything runs, adaptive replanning as steps complete, and a clear stopping condition (`plan_complete`). The tradeoff for that visibility and error recovery is more moving parts than a single ReAct loop.

## Step 6: Pattern -- Route-and-Solve ⭐

This section extends the [Function Calling Agent](01_function_calling.ipynb) pattern by introducing a router node that intelligently distributes queries to specialized subagents, each with their own grouped set of tools.

The Route-and-Solve architecture is ideal when you have a large number of tools that can be naturally grouped by category or domain. Instead of presenting all tools to a single agent, the router first determines which category of tools are needed, then routes the query to the appropriate subagent.

This approach offers several benefits:
- **Reduced tool selection errors**: Subagents only see relevant tools for their domain
- **Better scalability**: Easy to add new tool categories as subagents
- **Improved reasoning**: Specialized subagents can reason more effectively within their domain
- **Clear separation of concerns**: Tools are logically grouped by functionality

**Key concepts:**

1. **Router node with tool calling.** The router uses the LLM's native tool-calling capability to determine which subagent should handle the query. Each subagent is represented as an actual callable tool that the router can invoke directly.
2. **Subagents as callable tools.** Instead of just routing to a subagent name, each subagent is a proper tool that accepts the user query, executes the subagent's internal logic and tools, and returns the actual result.
3. **Subagent implementation.** Each subagent is a specialized function calling agent that only has access to tools in its category.
4. **Scalability.** Adding a new tool category is: define its tools, wrap the subagent as a tool, add it to the router's tool list -- no graph changes needed.

We use two categories -- **Finance Tools** and **Weather Tools** -- built from the `get_stock_price` and `get_current_weather` already imported at the top of this notebook, so there is no new tool code in this section at all.

### Define the agent state

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class RouteState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    current_subagent: str

### Create subagents

We create separate function calling agents for each tool category using `create_agent`. Each subagent only sees tools relevant to its domain.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

finance_tools = [get_stock_price]
finance_agent = create_agent(
    model=llm,
    tools=finance_tools,
)

weather_tools = [get_current_weather]
weather_agent = create_agent(
    model=llm,
    tools=weather_tools,
)

### Create router tools

Before creating the router node, we define wrapper functions that represent each subagent as a callable tool. These are what the router's tool-calling mechanism actually chooses between.

In [ ]:
from langchain.tools import tool

@tool
def finance_agent_tool(query: str) -> str:
    """
    Execute financial queries including stock prices, market data, and financial analysis.

    This tool executes a specialized finance agent that has access to stock price tools.

    Examples of queries this tool handles:
    - "What were the IBM stock prices on 2025-09-05?"
    - "Get me Microsoft stock data for today"
    - "Analyze Apple's recent stock performance"

    Use this tool when the user asks about:
    - Stock prices or historical data
    - Financial metrics and market information
    - Investment-related queries
    """
    state = {"messages": [HumanMessage(query)], "current_subagent": ""}
    result = finance_agent.invoke(state)
    return result["messages"][-1].content

@tool
def weather_agent_tool(query: str) -> str:
    """
    Execute weather queries including current conditions and forecasts.

    This tool executes a specialized weather agent that has access to weather tools.

    Examples of queries this tool handles:
    - "What is the weather in San Francisco?"
    - "Get me the current weather in London"
    - "What's the forecast for New York?"

    Use this tool when the user asks about:
    - Current weather conditions
    - Weather forecasts
    - Climate and atmospheric information
    """
    state = {"messages": [HumanMessage(query)], "current_subagent": ""}
    result = weather_agent.invoke(state)
    return result["messages"][-1].content

subagent_tools = [finance_agent_tool, weather_agent_tool]

Then we create the router node for the subagent tools.

In [ ]:
def router_node(state: RouteState) -> RouteState:
    """
    Router node that uses tool calling to determine which subagent to invoke.

    This node only invokes the LLM to generate tool calls. The actual tool execution
    is delegated to the ToolNode, which automatically handles detecting tool_calls,
    executing the appropriate subagent tool, and adding the ToolMessage with results.
    """
    messages = state["messages"]

    llm_with_tools = llm.bind_tools(subagent_tools)

    response = llm_with_tools.invoke(messages)

    messages = list(messages)
    messages.append(response)

    return RouteState(messages=messages, current_subagent="")

### Build the graph with a ToolNode

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import END

tool_node = ToolNode(tools=subagent_tools)

def end_node(state: RouteState) -> RouteState:
    """Returns the state with the final conversation; the router already added the response."""
    return state

def route_tools(state: RouteState) -> str:
    """Route to tools if the last message contains tool calls, otherwise go to end_node."""
    messages = state["messages"]
    last_message = messages[-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(RouteState)

graph_builder.add_node("router", router_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("end_node", end_node)

graph_builder.add_edge(START, "router")

graph_builder.add_conditional_edges(
    "router",
    route_tools,
    {
        "tools": "tools",
        END: "end_node",
    },
)

graph_builder.add_edge("tools", "router")

graph_builder.add_edge("end_node", END)

route_graph = graph_builder.compile()

### Try it

Now let's test the agent with various queries that will be routed to different subagents.

In [ ]:
def route_and_solve_agent(graph, user_input: str):
    """
    Run the Route and Solve agent with a user query: route it to the appropriate
    subagent, execute the subagent's tools as needed, and print each step.
    """
    user_message = HumanMessage(user_input)
    print(user_message.pretty_repr())

    input_state = {"messages": [user_message], "current_subagent": ""}

    for event in graph.stream(input_state):
        for node_name, node_state in event.items():
            if "messages" in node_state and node_state["messages"]:
                print(f"[{node_name}] {node_state['messages'][-1].pretty_repr()}")

A query with a ticker and a date should be routed to the finance subagent:

In [ ]:
route_and_solve_agent(route_graph, "What were the IBM stock prices on 2026-05-09?")

A weather query should be routed to the weather subagent:

In [ ]:
route_and_solve_agent(route_graph, "What is the weather in San Francisco?")

A query needing neither tool should be routed to neither subagent:

In [ ]:
route_and_solve_agent(route_graph, "What is the capital of France?")

### Simplified with `create_agent`

The Route and Solve pattern shown above can also be implemented more concisely using LangChain's `create_agent`, just like we did for the subagents themselves and in [`01_function_calling.ipynb`](01_function_calling.ipynb). This saves you from defining `router_node`, a `ToolNode`, and the graph by hand -- `create_agent` handles all of it internally, since a router choosing between subagent-tools is really just a function-calling agent whose tools happen to be other agents.

In [ ]:
router_agent = create_agent(
    model=llm,
    tools=subagent_tools,
)

route_and_solve_agent(router_agent, "What were the IBM stock prices on 2026-02-05?")
route_and_solve_agent(router_agent, "What is the weather in San Francisco?")
route_and_solve_agent(router_agent, "What is the capital of France?")

**Route-and-Solve recap:** adding a new domain is "define its tools, wrap it as one more subagent-tool" -- no graph changes. The tradeoff is an extra LLM call (the router) before any real work happens, and a router that only ever sees tool *names and descriptions*, not the underlying tools' own docstrings in detail.

## Step 7: Pattern -- ToolRAG ⭐

ToolRAG combines two abilities:

- Retrieval-Augmented Generation ([RAG](https://research.ibm.com/blog/retrieval-augmented-generation-RAG)): look up relevant information (here, tool descriptions) before answering.
- Tool use: choose and call the right tool.

In classic tool-calling, every tool's full definition goes into the LLM's prompt -- fine for a handful of tools, but it stops scaling once you have hundreds. ToolRAG pre-filters: a vector store retrieves only the tools that are semantically relevant to the current query, and only those get bound to the model. This section builds a small (5-tool) pool so the retrieval mechanism is easy to verify by eye, but the same approach is what scales to hundreds of tools in production.

### Initialize the embedding model

Alongside `llm` (already set up in Step 2), ToolRAG needs an embedding model to index tool descriptions. We use `ibm-granite/granite-embedding-small-english-r2` -- an IBM Granite family model, for consistency with the chat model, and small enough to run locally without a GPU.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(model_name=embeddings_model_path)

### Define a larger tool pool (demo tools)

For this proof of concept we use 5 tools, small enough that you can visually confirm the vector store is picking the right subset for a given query.

Two of these -- `get_stock_price_demo` and `get_weather_forecast_demo` -- are intentionally simplified mock tools with different signatures than the real `get_stock_price` / Plan-and-Solve's real `get_weather_forecast` used earlier in this notebook, so they're named distinctly to avoid confusion (and to avoid silently shadowing those earlier tools in this shared notebook).

In [ ]:
from langchain.tools import tool

@tool
def calculate_future_value(principal: float, rate: float, years: int) -> str:
    """Calculates the future value of an investment using compound interest."""
    future_value = principal * ((1 + rate) ** years)
    return f"The future value is ${future_value:.2f}."

@tool
def get_stock_price_demo(ticker: str) -> str:
    """Fetches the current or historical stock price for a given ticker symbol (e.g., IBM)."""
    if ticker == "IBM":
        return "The current price for IBM is $313.72."
    return f"Stock price for {ticker} not found in this mock-up."

@tool
def check_developer_skill(skill: str) -> str:
    """Checks the availability of an AI developer with a specific programming or ML skill."""
    if 'python' in skill.lower() or 'langchain' in skill.lower():
        return "Yes, we have multiple experienced developers with that skill set."
    return "Developer with that specific skill is currently limited."

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Converts a monetary amount between two specified currencies (e.g., USD to EUR)."""
    if from_currency == "USD" and to_currency == "EUR":
        converted = amount * 0.92
        return f"{amount} USD is approximately {converted:.2f} EUR."
    return f"Conversion from {from_currency} to {to_currency} is not supported."

@tool
def get_weather_forecast_demo(city: str) -> str:
    """Provides the current weather forecast for a specified city."""
    if 'boston' in city.lower():
        return "Boston's current weather is partly cloudy with a temperature of 5°C."
    return f"Weather data for {city} is not available in this mock-up."

toolrag_tools = [calculate_future_value, get_stock_price_demo, check_developer_skill, convert_currency, get_weather_forecast_demo]

print("=== All Initialized Tools ===")
for i, t in enumerate(toolrag_tools, start=1):
    print(f"  Tool {i}: '{t.name}': '{t.description}'")

tool_map = {t.name: t for t in toolrag_tools}

### Index the tools in a vector store

Before the agent can perform Tool-RAG, we need a searchable index of our tools:

- **Extracting tool metadata**: turn each tool's description into a `Document`, tagged with its name.
- **Embedding & storing**: vectorize descriptions with the embedding model, then store them in ChromaDB for semantic similarity search.

In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

tool_docs = [
    Document(page_content=t.description, metadata={"tool_name": t.name})
    for t in toolrag_tools
]

vectorstore = Chroma.from_documents(documents=tool_docs, embedding=embeddings_model)

print(f"Indexed {len(tool_docs)} tools in vectorstore.")

### Retrieve-then-bind agent graph

This vanilla LangGraph agent demonstrates the ToolRAG pattern end to end:

1. **Retrieve tools node**: semantically select relevant tools based on the query.
2. **LLM node**: dynamically binds *only* the retrieved tools to the LLM (reduces context bloat).
3. **Conditional edge**: routes to tools if calls detected, else END.
4. **Tool node**: executes calls using `ToolNode`.
5. **Loop back**: from tools to LLM for multi-turn reasoning.

The retrieval function itself bridges the vector store (semantic search over text) and the agent state (callable tool objects): it searches, then maps each result's `tool_name` back to the actual `BaseTool` via `tool_map`.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage, AIMessage
from langchain_core.tools import BaseTool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


def custom_retrieve_tools(query: str, limit: int = 5) -> list[BaseTool]:
    results = vectorstore.similarity_search_with_score(query, k=limit)
    retrieved_tools = [
        tool_map[doc.metadata['tool_name']]
        for doc, _ in results
        if doc.metadata['tool_name'] in tool_map
    ]
    print(f"[ToolRAG]: Retrieved {len(retrieved_tools)}: {[t.name for t in retrieved_tools]}")
    return retrieved_tools


class ToolRagState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    retrieved_tools: list[BaseTool]


def retrieve_tools_node(state: ToolRagState) -> ToolRagState:
    messages = state.get("messages", [])
    query = messages[-1].text
    retrieved_tools = custom_retrieve_tools(query, limit=3)
    return ToolRagState(retrieved_tools=retrieved_tools)


def toolrag_llm_node(state: ToolRagState) -> ToolRagState:
    messages = state.get("messages", [])
    retrieved_tools = state.get("retrieved_tools", [])
    bound_llm = llm.bind_tools(retrieved_tools)
    response = bound_llm.invoke(messages)
    return ToolRagState(messages=[response], retrieved_tools=retrieved_tools)


toolrag_tool_node = ToolNode(toolrag_tools)


def toolrag_should_continue(state: ToolRagState) -> str:
    messages = state.get("messages", [])
    last = messages[-1]
    return "tools" if isinstance(last, AIMessage) and last.tool_calls else END


workflow = StateGraph(state_schema=ToolRagState)
workflow.add_node("retrieve_tools", retrieve_tools_node)
workflow.add_node("llm", toolrag_llm_node)
workflow.add_node("tools", toolrag_tool_node)

workflow.set_entry_point("retrieve_tools")
workflow.add_edge("retrieve_tools", "llm")
workflow.add_conditional_edges("llm", toolrag_should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "llm")

toolrag_graph = workflow.compile()

try:
    toolrag_graph.get_graph().print_ascii()
except Exception:
    # Needs the optional `grandalf` dependency
    pass

### Try it

A query that needs three of the five tools -- watch the `[ToolRAG]` line to see which subset gets retrieved.

In [ ]:
from langchain_core.messages import HumanMessage

user_query = "What's the weather like in Boston, and can you check the stock price for IBM, then convert 100 USD to EUR?"
final_result = toolrag_graph.invoke(
    ToolRagState(messages=[HumanMessage(content=user_query)], retrieved_tools=[]),
)

final_message = final_result["messages"][-1]
print("\n--- Final Agent Response ---")
print(final_message.text)

### Further reading

We built this Tool-RAG agent from scratch with vanilla LangGraph; a few pre-built libraries abstract the same boilerplate (indexing, retrieval, dynamic binding) for production-scale tool pools:

- [langgraph-bigtool](https://github.com/langchain-ai/langgraph-bigtool): LangGraph extension for "Big Tool" agents; automates Tool-RAG with registry-based retrieval
- [LangChain Toolkits](https://docs.langchain.com/oss/javascript/integrations/tools): modular agent builders with RAG-infused toolkits
- [LlamaIndex Tool Integration](https://developers.llamaindex.ai/python/framework/module_guides/deploying/agents/tools/): index-based RAG for tools via `VectorStoreIndex`

**ToolRAG recap:** the model never sees a tool it wasn't retrieved for, so context stays bounded no matter how large the tool pool grows. The tradeoff is a new failure mode -- if retrieval misses the right tool, the model can't use it even if it "knows" the tool exists.

## Step 8: Pattern -- ReAct⭐

**ReAct** stands for **Reason + Act**. It is a design pattern that lets an LLM alternate between reasoning about what to do next ("Thought") and taking actions using external tools ("Action") until it reaches a final answer:

```
Thought → Action → Observation → Thought → ... → Answer
```

ReAct enables the model to adapt dynamically to intermediate results -- ideal when:
- The model must decide *which tools to use* and *in what order*.
- The task cannot be expressed as a simple, deterministic pipeline.
- The model benefits from iterative reasoning and external knowledge.

For deterministic workflows, such as simple API wrappers or rule-based steps, a fixed flow is still preferable. ReAct shines in cases where flexible, context-dependent decisions are needed.

This section builds a travel-planning assistant with three mock tools, first as an explicit hand-rolled loop (extending the one from [`01_function_calling.ipynb`](01_function_calling.ipynb), which never asked the model to show its reasoning), then the simpler way with `create_agent`.

### Define the tools

Tools are the LLM's interface to the digital world. To improve the likelihood of the right tool being selected, tools should have a clear name, a well-defined use case, a clear parameter schema, and return simple text the model can read as context for the next step. Developers should also keep toolsets narrow and non-overlapping.

Here we define three mock travel-related tools:
- `get_weather` — returns a climate description for a given city and month.
- `get_flight_info` — returns average flight prices.
- `get_hotel_prices` — returns average hotel ranges.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(destination: str, month: str) -> str:
    """Get weather for a destination and month."""
    return f"The weather in {destination} in {month} is cool and dry, highs ~16°C."

@tool
def get_flight_info(destination: str, month: str) -> str:
    """Get flight cost for a destination and month."""
    return f"Flights to {destination} in {month} average around $1200 USD round-trip."

@tool
def get_hotel_prices(destination: str, month: str) -> str:
    """Get hotel prices for a destination and month."""
    return f"Hotels in {destination} range from $100 to $300 per night in {month}."

react_tools = [get_weather, get_flight_info, get_hotel_prices]

tool_descriptions = "\n".join([f"- {t.name}: {t.description}" for t in react_tools])
tool_names = ", ".join([t.name for t in react_tools])

### Write the ReAct system instructions

Next, we craft a **system prompt** that teaches the model the ReAct workflow: it defines the available tools, demonstrates the *Thought → Action → Observation → Answer* pattern, specifies strict formatting rules so only one tool is called at a time, and instructs the model to wait for an Observation before continuing. This explicit, step-by-step example helps smaller models like Granite 4.2 stay consistent and avoids hallucinating multiple actions at once.

In [ ]:
react_instructions = f"""
You are a helpful travel planning assistant that uses the ReAct (Reasoning + Acting) framework to answer travel-related questions.

Always reason step by step in a visible loop of **Thought → Action  → Observation**.
When finished, output the **Answer**.

Tool Usage
Available Tools to you:
{tool_descriptions}

You MUST use the following format to answer the user's travel related question, using the list of tools as needed:

Thought: <Explain thoughts and action to take>
Action: {{"name": "tool_name", "args": {{"destination": "...", "month":"..."}}}}
Observation: (filled later with the tool result)
(This loop may repeat multiple times, until you know the final answer)
Thought: I know the final answer.
Answer: <final answer to the user>

Example Format

User: I want to visit Paris in May. How much should that cost?
Thought: The user wants travel advice. I should check the hotel prices
Action: {{"name": "get_hotel_prices", "args": {{"destination": "Paris", "month": "May"}}
Observation: Hotel prices are between $250-500.
Thought: Next I need to check the price of flights
Action: {{"name": "get_flight_info, "args": {{"destination": "Paris", "month": "May"}}
Observation: Flight to Paris in May are $500.
Thought: I know the final answer.
Answer: Travel to Paris is good this time of year. Hotel prices are between $250-500 and flight tickets are around $500


Rules
* Always show each step beginning with "Thought:" on it's own line BEFORE performing any actions.
* YOU MUST make only ONE Action at a time per loop
* NEVER propose multiple Actions together.
* After performing ONE Action, WAIT for the Observation before reasoning again
* DO NOT Write Observation yourself. Only the system will provide Observation after your Action
* Only use tools listed in "Available Tools", which are the following: {tool_names}
* Stop ONLY when you reach the final Answer
* NEVER include Final Answer in the same turn as an Action. If you just took an Action, wait for an Observation first.
* If you think no Action is needed, just output Answer, do not generate a JSON Action.
* Only use tools these tools: {tool_names}
"""

### Implement the ReAct loop by hand

This is a lightweight, framework-agnostic implementation of the ReAct pattern, in the same spirit as the hand-rolled tool loop from [`01_function_calling.ipynb`](01_function_calling.ipynb) -- except the model is now asked to show its reasoning (`Thought:`) before each action, and we parse that text instead of relying only on structured `tool_calls`.

Each iteration:
1. Sends the current messages (system + chat history) to the model.
2. Extracts any JSON action object (`{"name": "...", "args": {...}}`).
3. Executes the selected tool and captures its result as an **Observation**.
4. Appends the Observation to the message history.
5. Repeats until the model outputs a final **Answer**.

Because we're parsing the model's own text for `Action:`/`Answer:` markers, we ask `get_llm()` for a `stop` sequence at `"Observation:"` -- otherwise the model might write its own fake Observation instead of waiting for the real tool result. This is the one place in this notebook where a pattern needs a differently-configured `llm` than the shared one from Step 2.

In [ ]:
llm_react = get_llm(stop=["Observation:", "\nObservation:", " Observation:"])
print(f"Connected via {type(llm_react).__name__}, model={getattr(llm_react, 'model', None)!r}")

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, AIMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ReactState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
import json

def call_model(state: ReactState):
    """Single ReAct reasoning step without regex loops.
    Extracts everything between the first '{' and the last '}' after Action:.
    """
    messages: list[AnyMessage] = [SystemMessage(content=react_instructions), *state.get("messages", [])]
    ai = llm_react.invoke(messages)
    output_text = (ai.text or "").strip()
    print(output_text)

    if "Action:" in output_text:
        raw_action = output_text.split("Action:", 1)[-1]
        start = raw_action.find("{")
        end = raw_action.rfind("}") + 1
        if start != -1 and end != -1:
            raw_action = raw_action[start:end]
        else:
            raw_action = None

        if raw_action:
            try:
                # Clean up single quotes if model uses them
                cleaned = raw_action.replace("'", '"')
                action = json.loads(cleaned)

                tool_name = action.get("name")
                args = action.get("args", {}) or {}
                react_tool = next((t for t in react_tools if t.name == tool_name), None)

                if react_tool:
                    obs = react_tool.invoke(args)
                    print(f"Observation: {obs}")
                    return {"messages": [ai, ("user", f"Observation: {obs}")]}
                else:
                    print(f"Unknown tool: {tool_name}")
            except Exception as e:
                print(f"Failed to parse Action JSON: {e}")
                return {"messages": [ai]}  # fail-safe end

    if "Answer:" in output_text:
        print("Final Answer reached.")
        return {"messages": [ai]}

    print("No Action/Answer detected; ending to avoid loop.")
    return {"messages": [ai]}


def route_from_llm(state: ReactState):
    messages = state.get("messages", [])
    last = messages[-1]
    if isinstance(last, AIMessage):
        # Final answer, or no Action and no Answer -> end either way (prevents infinite loop)
        return "end"
    else:
        # A tuple like ("user", "Observation: ...") -> continue to llm
        return "llm"

### Run it

Now let's test our **travel-planning assistant** with a sample query. The model should think through the problem, choose a tool, wait for the Observation, and continue until it produces a final Answer summarizing all gathered details. Try adjusting the city or month to see how the model dynamically changes which tools it calls and in what order.

In [ ]:
react_graph = StateGraph(ReactState)
react_graph.add_node("llm", call_model)
react_graph.add_edge(START, "llm")
react_graph.add_conditional_edges("llm", route_from_llm, {"llm": "llm", "end": END})
react_app = react_graph.compile()

In [ ]:
inputs = ReactState(messages=[HumanMessage(content="I want to visit Tokyo next April. What should I expect about weather and costs?")])
result = react_app.invoke(inputs)

### The simpler way: `create_agent`

Earlier, we built the full ReAct loop manually -- parsing model outputs, calling tools, and feeding back Observations step by step. That approach helped illustrate *how* ReAct works, but it required a lot of orchestration code.

With LangChain's modern API, `create_agent()` builds a ready-to-use reasoning agent that runs the entire Thought → Action → Observation → Answer loop automatically -- the same behavior as before but with far less code.

A note about the `system_prompt` argument: here we provide the same `react_instructions` used above to obtain the same ReAct-with-visible-reasoning behavior. If we omit it, the agent behaves like the plain function-calling agent from [`01_function_calling.ipynb`](01_function_calling.ipynb) instead -- no visible `Thought:`/`Action:` text, just tool calls. Because `create_agent` manages the loop itself (it doesn't rely on parsing "Observation:" out of the model's text), we don't need the `stop` sequence here -- a fresh, plain `llm` is enough.

In [ ]:
llm_react_simple = get_llm()
print(f"Connected via {type(llm_react_simple).__name__}, model={getattr(llm_react_simple, 'model', None)!r}")

react_agent = create_agent(
    model=llm_react_simple,
    tools=react_tools,
    system_prompt=react_instructions,
)

response = react_agent.invoke({
    "messages": [HumanMessage(content="I want to visit Tokyo next month, which will be April. What should I expect in terms of weather and cost?")]
})

print(response["messages"][-1].text)

**ReAct recap:** the visible `Thought:` trace makes the model's reasoning inspectable, which is valuable for debugging and for building attendee trust in *why* the agent did something -- not just *what* it did. `ReAct` is one of the most widely adopted agentic patterns for exactly that reason, but it's only one approach; the three patterns above solve different problems (structured multi-step plans, tool-domain separation, and tool-set scale) that a single ReAct loop doesn't address by itself.

## Reuse in later notebooks

This notebook didn't add anything new to `src/granite_agent/` -- every pattern above reused `get_llm()`, `get_stock_price` and `get_current_weather` from [`01_function_calling.ipynb`](01_function_calling.ipynb) rather than extending the shared package. What's reused, from where:

| Reusable | From |
| --- | --- |
| `granite_agent.model.get_llm()` | 01, Step 2 -- used, with different arguments, in every pattern section |
| `granite_agent.tools.get_stock_price`, `granite_agent.tools.get_current_weather` | 01, Step 4 -- used as-is in Route-and-Solve, and `get_current_weather` in Plan-and-Solve |

Each pattern's own tools, prompts, state and graph are specific to that pattern and stay local to this notebook.